**<h1 align="center">Dataset Construction (JobHop × ESCO)</h1>**

This notebook constructs the final datasets used in the thesis by integrating ESCO job information with the JobHop career trajectory dataset. ESCO occupation and skill data are first combined to create job descriptions enriched with essential skills.

The JobHop dataset is then loaded (train/validation/test splits) and preprocessed independently per split to avoid data leakage. Career histories are organized into chronological timelines, and CV representations are built from past occupations and associated ESCO skills.

The next observed occupation defines the prediction target, and ESCO job descriptions are aligned to these targets via normalized label matching. The final outputs are train, validation, and test datasets ready for modeling and evaluation.

To reproduce the dataset used in the experiments, update the data directory at the end of the notebook and run the notebook to generate and save the processed datasets.

**ESCO dataset - v1.2.1 / classification / en / csv** available at: https://esco.ec.europa.eu/en/use-esco/download

**JobHop** dataset available at: https://huggingface.co/datasets/aida-ugent/JobHop


<a class="anchor" id="chapter1"></a>

# 1. Imports

</a>

In [1]:
import pandas as pd
import numpy as np
import os
import hashlib
import random

from datasets import load_dataset

/opt/anaconda3/envs/ResumeMatcherThesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [13]:
dictionary = pd.read_csv("../Data/ESCO_Files/dictionary_en.csv")
dictionary.head()

,filename,data header,property,description
0,occupations,conceptType,NaN,No direct related property exists
1,occupations,conceptUri,NaN,No direct related property exists
2,occupations,iscoGroup,http://www.w3.org/2004/02/skos/core#notation,"A notation, also known as classification code,..."
3,occupations,preferredLabel,http://www.w3.org/2004/02/skos/core#prefLabel,"The preferred lexical label for a resource, in..."
4,occupations,altLabels,http://www.w3.org/2004/02/skos/core#altLabel,An alternative lexical label for a resource.


In [17]:
print(dictionary["filename"].unique())

['occupations' 'skills' 'ISCOGroups' 'skillGroups'
 'transversalSkillsCollection' 'languageSkillsCollection' 'conceptSchemes'
 'digCompSkillsCollection' 'greenSkillsCollection'
 'researchSkillsCollection' 'digitalSkillsCollection'
 'researchOccupationsCollection' 'filename' 'broaderRelationsOccPillar'
 'broaderRelationsSkillPillar' 'occupationSkillRelations'
 'skillSkillRelations' 'skillsHierarchy']


In [18]:
# ESCO FILES
occ = pd.read_csv("../Data/ESCO_Files/occupations_en.csv")
occrs = pd.read_csv("../Data/ESCO_Files/occupationSkillRelations_en.csv")
skills = pd.read_csv("../Data/ESCO_Files/skills_en.csv")

In [ ]:
# JOBHOP (requires Hugging Face login and access token)
ds = load_dataset("aida-ugent/JobHop")

train_ds = ds["train"]
val_ds = ds["validation"]
test_ds = ds["test"]

train_df = train_ds.to_pandas()
val_df = val_ds.to_pandas()
test_df = test_ds.to_pandas()

<a class="anchor" id="chapter3"></a>

# 3. ESCO Initial Analysis

</a>

<a class="anchor" id="sub-section-3_1"></a>

## 3.1. Occupations

</a>

<a class="anchor" id="sub-section-3_1_1"></a>

### 3.1.1. Types & Structure

</a>

In [20]:
dictionary[dictionary["filename"] == "occupations"]

,filename,data header,property,description
0,occupations,conceptType,NaN,No direct related property exists
1,occupations,conceptUri,NaN,No direct related property exists
2,occupations,iscoGroup,http://www.w3.org/2004/02/skos/core#notation,"A notation, also known as classification code,..."
3,occupations,preferredLabel,http://www.w3.org/2004/02/skos/core#prefLabel,"The preferred lexical label for a resource, in..."
4,occupations,altLabels,http://www.w3.org/2004/02/skos/core#altLabel,An alternative lexical label for a resource.
5,occupations,hiddenLabels,http://www.w3.org/2004/02/skos/core#hiddenLabel,A lexical label for a resource that should be ...
6,occupations,status,http://purl.org/iso25964/skos-thes#status,ISO status\n- on ThesaurusConcept\n- on Thesau...
7,occupations,modifiedDate,http://purl.org/dc/terms/modified,Date on which the resource was changed.
8,occupations,regulatedProfessionNote,http://data.europa.eu/esco/model#regulatedProf...,The subject occupation is regulated according ...
9,occupations,scopeNote,http://www.w3.org/2004/02/skos/core#scopeNote,A note that helps to clarify the meaning and/o...


In [21]:
print("Shape of Occupations df:", occ.shape)
occ.head()

Shape of Occupations df: (3043, 15)


,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,naceCode
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,director of technical arts\ntechnical supervis...,NaN,released,2024-01-25T11:28:50.295Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Technical directors realise the artistic visio...,2654.1.7,http://data.europa.eu/ux2/nace2.1/9031
1,Occupation,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,wire drawer\nforming machine operative\ndraw m...,NaN,released,2024-01-23T10:09:32.099Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Metal drawing machine operators set up and ope...,8121.4,http://data.europa.eu/ux2/nace2.1/242
2,Occupation,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,precision device quality control supervisor\np...,NaN,released,2024-01-25T15:00:12.188Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Precision device inspectors make sure precisio...,7543.10.3,http://data.europa.eu/ux2/nace2.1/2651
3,Occupation,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,air traffic safety electronics hardware specia...,NaN,released,2024-01-29T16:01:13.998Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Air traffic safety technicians provide technic...,3155.1,http://data.europa.eu/ux2/nace2.1/5223
4,Occupation,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,yield manager\nhospitality yields manager\nhos...,NaN,released,2024-01-11T10:28:45.871Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Hospitality revenue managers maximise revenue ...,2431.9,"http://data.europa.eu/ux2/nace2.1/701,\nhttp:/..."


In [22]:
occ.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3043 entries, 0 to 3042
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   conceptType              3043 non-null   object
 1   conceptUri               3043 non-null   object
 2   iscoGroup                3043 non-null   int64 
 3   preferredLabel           3043 non-null   object
 4   altLabels                3043 non-null   object
 5   hiddenLabels             8 non-null      object
 6   status                   3043 non-null   object
 7   modifiedDate             3043 non-null   object
 8   regulatedProfessionNote  3043 non-null   object
 9   scopeNote                310 non-null    object
 10  definition               8 non-null      object
 11  inScheme                 3043 non-null   object
 12  description              3043 non-null   object
 13  code                     3043 non-null   object
 14  naceCode                 3043 non-null  

<a class="anchor" id="sub-section-3_1_2"></a>

### 3.1.2. Duplicates

</a>

In [23]:
occ[occ.index.duplicated() == True]

,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,naceCode


In [24]:
duplicates = occ.duplicated().sum()
print(f"There are {duplicates} duplicated rows.")

There are 0 duplicated rows.


<a class="anchor" id="sub-section-3_1_3"></a>

### 3.1.3. Missing Values

</a>

In [25]:
occ.isna().sum()

conceptType                   0
conceptUri                    0
iscoGroup                     0
preferredLabel                0
altLabels                     0
hiddenLabels               3035
status                        0
modifiedDate                  0
regulatedProfessionNote       0
scopeNote                  2733
definition                 3035
inScheme                      0
description                   0
code                          0
naceCode                      0
dtype: int64

<a class="anchor" id="sub-section-3_1_4"></a>

### 3.1.4. Statistical Analysis

</a>

In [26]:
occ.describe(include = object).T

,count,unique,top,freq
conceptType,3043,1,Occupation,3043
conceptUri,3043,3039,http://data.europa.eu/esco/occupation/4d27152a...,2
preferredLabel,3043,3039,early years teaching assistant,2
altLabels,3043,3039,reception teaching assistant\nassistant in ear...,2
hiddenLabels,8,8,sexual health consultant\nyouth policy manager...,1
status,3043,1,released,3043
modifiedDate,3043,2902,2025-06-25T07:45:00.959Z,27
regulatedProfessionNote,3043,2,http://data.europa.eu/esco/regulated-professio...,3027
scopeNote,310,274,Excludes people performing managerial activities.,12
definition,8,8,Excludes choreologist.,1


<a class="anchor" id="sub-section-3_1_5"></a>

### 3.1.5. Drop Columns

</a>

Columns used to construct the job descriptions:
- conceptUri: ESCO occupation ID
- isoGroup: ESCO code
- preferredLabel: Job title
- description: Job description text

In [27]:
print(list(occ))

['conceptType', 'conceptUri', 'iscoGroup', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'description', 'code', 'naceCode']


In [28]:
occ = occ.drop(['conceptType', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 
                'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'code', 
                'naceCode'], axis=1)

In [29]:
occ = occ.rename(columns={"conceptUri": "occupationUri"})

In [30]:
occ.head()

,occupationUri,iscoGroup,preferredLabel,description
0,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,Technical directors realise the artistic visio...
1,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,Metal drawing machine operators set up and ope...
2,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,Precision device inspectors make sure precisio...
3,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,Air traffic safety technicians provide technic...
4,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,Hospitality revenue managers maximise revenue ...


<a class="anchor" id="sub-section-3_2"></a>

## 3.2. Occupation Skill Relations

</a>

<a class="anchor" id="sub-section-3_2_1"></a>

### 3.2.1. Types & Structure

</a>

In [31]:
dictionary[dictionary["filename"] == "occupationSkillRelations"]

,filename,data header,property,description
135,occupationSkillRelations,occupationUri,NaN,No direct related property exists
136,occupationSkillRelations,occupationLabel,http://www.w3.org/2004/02/skos/core#prefLabel,"The preferred lexical label for a resource, in..."
137,occupationSkillRelations,relationType,http://data.europa.eu/esco/model#relatedOption...,The ESCO skill or competence that is relevant ...
138,occupationSkillRelations,skillType,http://data.europa.eu/esco/model#skillType,Type of competence (a tagging concept)
139,occupationSkillRelations,skillUri,NaN,No direct related property exists
140,occupationSkillRelations,skillLabel,http://www.w3.org/2004/02/skos/core#prefLabel,"The preferred lexical label for a resource, in..."


In [32]:
print("Shape of Occupation Skill Relations df:", occrs.shape)
occrs.head()

Shape of Occupation Skill Relations df: (126051, 6)


,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel
0,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,knowledge,http://data.europa.eu/esco/skill/fed5b267-73fa...,theatre techniques
1,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/05bc7677-5a64...,organise rehearsals
2,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/271a36a0-bc7a...,write risk assessment on performing arts produ...
3,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/47ed1d37-971b...,coordinate with creative departments
4,http://data.europa.eu/esco/occupation/00030d09...,technical director,essential,skill/competence,http://data.europa.eu/esco/skill/591dd514-735b...,adapt to artists' creative demands


In [33]:
occrs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126051 entries, 0 to 126050
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   occupationUri    126051 non-null  object
 1   occupationLabel  126051 non-null  object
 2   relationType     126051 non-null  object
 3   skillType        125992 non-null  object
 4   skillUri         126051 non-null  object
 5   skillLabel       126051 non-null  object
dtypes: object(6)
memory usage: 5.8+ MB


<a class="anchor" id="sub-section-3_2_2"></a>

### 3.2.2. Duplicates

</a>

In [34]:
occrs[occrs.index.duplicated() == True]

,occupationUri,occupationLabel,relationType,skillType,skillUri,skillLabel


In [35]:
duplicates = occrs.duplicated().sum()
print(f"There are {duplicates} duplicated rows.")

There are 0 duplicated rows.


<a class="anchor" id="sub-section-3_2_3"></a>

### 3.2.3. Missing Values

</a>

In [36]:
occrs.isna().sum()

occupationUri       0
occupationLabel     0
relationType        0
skillType          59
skillUri            0
skillLabel          0
dtype: int64

<a class="anchor" id="sub-section-3_2_4"></a>

### 3.2.4. Statistical Analysis

</a>

In [37]:
occrs.describe(include = object).T

,count,unique,top,freq
occupationUri,126051,3039,http://data.europa.eu/esco/occupation/9b889f07...,178
occupationLabel,126051,3039,specialised doctor,178
relationType,126051,2,essential,67600
skillType,125992,2,skill/competence,91608
skillUri,126051,13475,http://data.europa.eu/esco/skill/415abd43-e8e5...,340
skillLabel,126051,13475,use different communication channels,340


In [38]:
occrs['relationType'].unique()

array(['essential', 'optional'], dtype=object)

In [39]:
# Filter to only essential skills
essential_skills = occrs[occrs['relationType'] == 'essential']

In [40]:
essential_skills['relationType'].describe()

count         67600
unique            1
top       essential
freq          67600
Name: relationType, dtype: object

In [41]:
# Group by skill labels by occupationUri
occ_to_skills = (
    essential_skills
    .groupby('occupationUri')['skillLabel']
    .apply(list)
    .reset_index(name='essential_skills')
)

In [42]:
print(occ_to_skills.shape)
occ_to_skills.head()

(3039, 2)


,occupationUri,essential_skills
0,http://data.europa.eu/esco/occupation/00030d09...,"[theatre techniques, organise rehearsals, writ..."
1,http://data.europa.eu/esco/occupation/000e93a3...,"[quality and cycle time optimisation, types of..."
2,http://data.europa.eu/esco/occupation/0019b951...,"[quality assurance procedures, precision engin..."
3,http://data.europa.eu/esco/occupation/0022f466...,"[aircraft flight control systems, safety engin..."
4,http://data.europa.eu/esco/occupation/002da35b...,"[property management software, produce statist..."


In [43]:
occ_to_skills['essential_skills'].apply(len).describe()

count    3039.000000
mean       22.244159
std        11.991231
min         3.000000
25%        14.000000
50%        19.000000
75%        27.000000
max        99.000000
Name: essential_skills, dtype: float64

There are 3043 occupations in the Occupations dataframe, while in this dataframe there are only 3039. 4 occupations with no essential skills.

<a class="anchor" id="sub-section-3_3"></a>

## 3.3. Merge Occupations and Occupation Skill Relations

</a>

In [44]:
# Merge occupations and occupation skill relations on occupationUri
jd_df = occ.merge(occ_to_skills, on='occupationUri', how='left')

In [45]:
print("Shape: ", jd_df.shape)
jd_df.head()

Shape:  (3043, 5)


,occupationUri,iscoGroup,preferredLabel,description,essential_skills
0,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,Technical directors realise the artistic visio...,"[theatre techniques, organise rehearsals, writ..."
1,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,Metal drawing machine operators set up and ope...,"[quality and cycle time optimisation, types of..."
2,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,Precision device inspectors make sure precisio...,"[quality assurance procedures, precision engin..."
3,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,Air traffic safety technicians provide technic...,"[aircraft flight control systems, safety engin..."
4,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,Hospitality revenue managers maximise revenue ...,"[property management software, produce statist..."


In [46]:
jd_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3043 entries, 0 to 3042
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   occupationUri     3043 non-null   object
 1   iscoGroup         3043 non-null   int64 
 2   preferredLabel    3043 non-null   object
 3   description       3043 non-null   object
 4   essential_skills  3043 non-null   object
dtypes: int64(1), object(4)
memory usage: 119.0+ KB


In [47]:
jd_df[jd_df.index.duplicated() == True]

,occupationUri,iscoGroup,preferredLabel,description,essential_skills


*Note: cannot check duplicate rows because essential_skills contains a list*

In [48]:
jd_df['essential_skills'].apply(len).describe()

count    3043.000000
mean       22.237266
std        11.987292
min         3.000000
25%        14.000000
50%        19.000000
75%        27.000000
max        99.000000
Name: essential_skills, dtype: float64

Structure: **Job Title. Job Description. Essential skills: list of skills.**

In [ ]:
# Initialize job description text using occupation label and description
jd_df["jd_text"] = jd_df.apply(
    lambda row: f"{row['preferredLabel']}. {row['description']}",
    axis=1
)

# Append essential skills (if present) as a semicolon-separated list
jd_df.loc[jd_df["essential_skills"].astype(bool), "jd_text"] += (
    " Essential skills: " +
    jd_df["essential_skills"].apply(lambda x: "; ".join(x)) + "."
)

In [50]:
jd_df['jd_text'].head()

0    technical director. Technical directors realis...
1    metal drawing machine operator. Metal drawing ...
2    precision device inspector. Precision device i...
3    air traffic safety technician. Air traffic saf...
4    hospitality revenue manager. Hospitality reven...
Name: jd_text, dtype: object

<a class="anchor" id="chapter4"></a>

# 4. JobHop Initial Analysis

</a>

<a class="anchor" id="sub-section-4_1"></a>

## 4.1. Types & Structure

</a>

In [51]:
print("Shape of train df:", train_df.shape)
train_df.head(10)

Shape of train df: (1341832, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,0,resource manager,Resource managers manage resources for all pot...,1324.8.3,Q1 2016,Q2 2019,True
1,0,health and safety officer,Health and safety officers execute plans for t...,2263.3,Q1 2017,Q2 2019,True
2,0,integration engineer,Integration engineers develop and implement so...,2511.17,Q1 2013,Q1 2016,True
3,0,programme manager,Programme managers coordinate and oversee seve...,1213.4,Q2 2012,Q1 2013,True
4,0,product development engineering drafter,Product development engineering drafters desig...,3118.3.12,Q1 2011,Q2 2012,True
5,0,move manager,Move managers coordinate all the resources and...,1324.4,Q1 2006,Q3 2008,True
6,0,customer contact centre information clerk,Customer contact centre information clerks pro...,4222.1,Q1 2004,Q4 2007,True
7,0,ICT help desk manager,ICT help desk managers monitor the delivery of...,3512.2,Q1 2002,Q1 2004,True
8,0,ICT help desk agent,ICT help desk agents provide technical assista...,3512.1,Q1 2000,Q1 2002,True
9,0,language school teacher,Language school teachers educate non-age-speci...,2353.1,Q3 1996,Q1 2000,True


In [52]:
print("Shape of validation df:", val_df.shape)
val_df.head()

Shape of validation df: (167945, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,5,medical administrative assistant,Medical administrative assistants work very cl...,3344.1,Q1 2013,Q4 2013,True
1,5,administrative assistant,Administrative assistants provide administrati...,3343.1,Q4 2013,Q4 2018,True
2,5,room attendant,"Room attendants clean, tidy and restock guest ...",9112.4,None,None,True
3,6,unknown,unknown,None,Q1 2010,Q4 2011,True
4,6,unknown,unknown,None,Q1 2009,None,True


In [53]:
print("Shape of test df:", test_df.shape)
test_df.head()

Shape of test df: (167924, 7)


,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,16,psychology lecturer,"Psychology lecturers are subject professors, t...",2310.1.35,Q1 2012,Q2 2012,True
1,16,human resources assistant,Human resources assistants provide support in ...,4416.1,Q1 2009,Q3 2011,True
2,16,human resources assistant,Human resources assistants provide support in ...,4416.1,Q4 2003,Q4 2008,True
3,16,human resources manager,"Human resources managers plan, design and impl...",1212.2,Q3 2000,Q4 2003,True
4,36,leaflet distributor,"Leaflet distributors hand out flyers, leaflet ...",9510.1,Q1 2005,Q4 2006,False


In [54]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1341832 entries, 0 to 1341831
Data columns (total 7 columns):
 #   Column               Non-Null Count    Dtype 
---  ------               --------------    ----- 
 0   person_id            1341832 non-null  int64 
 1   matched_label        1341832 non-null  object
 2   matched_description  1341832 non-null  object
 3   matched_code         1330799 non-null  object
 4   start_date           1187091 non-null  object
 5   end_date             1043416 non-null  object
 6   university_studies   1341832 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 62.7+ MB


In [55]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167945 entries, 0 to 167944
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   person_id            167945 non-null  int64 
 1   matched_label        167945 non-null  object
 2   matched_description  167945 non-null  object
 3   matched_code         166430 non-null  object
 4   start_date           148683 non-null  object
 5   end_date             130359 non-null  object
 6   university_studies   167945 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 7.8+ MB


In [56]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167924 entries, 0 to 167923
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   person_id            167924 non-null  int64 
 1   matched_label        167924 non-null  object
 2   matched_description  167924 non-null  object
 3   matched_code         166596 non-null  object
 4   start_date           149029 non-null  object
 5   end_date             131111 non-null  object
 6   university_studies   167924 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 7.8+ MB


<a class="anchor" id="sub-section-4_2"></a>

## 4.2. Duplicates

</a>

In [57]:
print(train_df[train_df.index.duplicated() == True])
print(val_df[val_df.index.duplicated() == True])
print(test_df[test_df.index.duplicated() == True])

Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []
Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []
Empty DataFrame
Columns: [person_id, matched_label, matched_description, matched_code, start_date, end_date, university_studies]
Index: []


In [58]:
print("Train: ", train_df.duplicated().mean())
print("Validation: ", val_df.duplicated().mean())
print("Test: ", test_df.duplicated().mean())

Train:  0.02633712715153611
Validation:  0.02657417606954658
Test:  0.02564850765822634


In [60]:
print(f"There are {train_df.duplicated().sum()} duplicated rows in train_df.")
print(f"There are {val_df.duplicated().sum()} duplicated rows in val_df.")
print(f"There are {test_df.duplicated().sum()} duplicated rows in test_df.")

There are 35340 duplicated rows in train_df.
There are 4463 duplicated rows in val_df.
There are 4307 duplicated rows in test_df.


In [61]:
print("Train: ", train_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())
print("Validation: ", val_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())
print("Test: ", test_df.duplicated(subset=['person_id', 'matched_code', 'start_date', 'end_date']).mean())

Train:  0.02633712715153611
Validation:  0.02657417606954658
Test:  0.02564850765822634


In [ ]:
# Drop duplicates
train_df = train_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])
val_df = val_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])
test_df = test_df.drop_duplicates(subset=['person_id', 'matched_code', 'start_date', 'end_date'])

<a class="anchor" id="sub-section-4_3"></a>

## 4.3. Missing Values

</a>

In [63]:
train_df.isna().sum()

person_id                   0
matched_label               0
matched_description         0
matched_code            10223
start_date             134229
end_date               274907
university_studies          0
dtype: int64

In [64]:
val_df.isna().sum()

person_id                  0
matched_label              0
matched_description        0
matched_code            1377
start_date             16643
end_date               34551
university_studies         0
dtype: int64

In [65]:
test_df.isna().sum()

person_id                  0
matched_label              0
matched_description        0
matched_code            1250
start_date             16422
end_date               33955
university_studies         0
dtype: int64

<a class="anchor" id="sub-section-4_4"></a>

## 4.4. Statistical Analysis

</a>

In [66]:
train_df.describe(include = object).T

,count,unique,top,freq
matched_label,1306492,2955,unknown,210964
matched_description,1306492,2955,unknown,210964
matched_code,1296269,2955,unknown,200741
start_date,1172263,227,Q1 2011,61816
end_date,1031585,216,Q4 2012,59030


In [67]:
val_df.describe(include = object).T

,count,unique,top,freq
matched_label,163482,2606,unknown,26531
matched_description,163482,2606,unknown,26531
matched_code,162105,2606,unknown,25154
start_date,146839,203,Q1 2011,7711
end_date,128931,190,Q4 2012,7381


In [68]:
test_df.describe(include = object).T

,count,unique,top,freq
matched_label,163617,2619,unknown,26432
matched_description,163617,2619,unknown,26432
matched_code,162367,2619,unknown,25182
start_date,147195,203,Q1 2011,7639
end_date,129662,194,Q4 2013,7312


In [ ]:
# Code
print("Unknown rate for train: ", (train_df['matched_code'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_code'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_code'] == 'unknown').mean())

Unknown rate for train:  0.15364885510205956
Unknown rate for validation:  0.15386403396092535
Unknown rate for test:  0.15390821247180916


In [70]:
# Label
print("Unknown rate for train: ", (train_df['matched_label'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_label'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_label'] == 'unknown').mean())

Unknown rate for train:  0.16147362555606923
Unknown rate for validation:  0.16228697960631752
Unknown rate for test:  0.16154800540286157


These rows do not have a valid ESCO occupation mapping. The original resume text could not be aligned to any ESCO occupation, therefore: no occupation, no ESCO skills, no ontology relations. These rows are structurally unusable for ESCO-augmented CV–JD matching.

In [71]:
# Remove rows with unresolved occupation mappings ("unknown")
train_df = train_df[(train_df["matched_code"] != "unknown") & (train_df["matched_label"] != "unknown")].copy()
val_df = val_df[(val_df["matched_code"] != "unknown") & (val_df["matched_label"] != "unknown")].copy()
test_df = test_df[(test_df["matched_code"] != "unknown") & (test_df["matched_label"] != "unknown")].copy()

In [72]:
print("Unknown rate for train: ", (train_df['matched_code'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_code'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_code'] == 'unknown').mean())

Unknown rate for train:  0.0
Unknown rate for validation:  0.0
Unknown rate for test:  0.0


In [73]:
print("Unknown rate for train: ", (train_df['matched_label'] == 'unknown').mean())
print("Unknown rate for validation: ", (val_df['matched_label'] == 'unknown').mean())
print("Unknown rate for test: ", (test_df['matched_label'] == 'unknown').mean())

Unknown rate for train:  0.0
Unknown rate for validation:  0.0
Unknown rate for test:  0.0


In [74]:
print("Train shape: ", train_df.shape)
train_df.describe(include = object).T

Train shape:  (1095528, 7)


,count,unique,top,freq
matched_label,1095528,2954,administrative assistant,56538
matched_description,1095528,2954,Administrative assistants provide administrati...,56538
matched_code,1095528,2954,3343.1,56538
start_date,962862,224,Q1 2011,50775
end_date,824954,212,Q4 2012,46531


<a class="anchor" id="chapter5"></a>

# 5. Creating Dataset

</a>

<a class="anchor" id="sub-section-5_1"></a>

## 5.1. Logic Check

</a>

**Check people with less than 2 experiences**

In [75]:
train_df.groupby('person_id')['matched_code'].nunique().describe()

count    276692.000000
mean          3.330273
std           1.910435
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max          18.000000
Name: matched_code, dtype: float64

In [76]:
print("People with ≥ 2 occupations in train: ", (train_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())
print("People with ≥ 2 occupations in val: ", (val_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())
print("People with ≥ 2 occupations in test: ", (test_df.groupby('person_id')['matched_code'].nunique() >= 2).mean())

People with ≥ 2 occupations in train:  0.830381796365634
People with ≥ 2 occupations in val:  0.8303811367642384
People with ≥ 2 occupations in test:  0.8304957363780893


For modeling and evaluation, the sample should be restricted to individuals with at least two distinct occupations, as career transitions are required to construct CV–job pairs.

In [77]:
# Keep only individuals with at least two distinct occupations (valid transitions)
def valid_persons(df):
    return df.groupby('person_id')['matched_code'].nunique().loc[lambda x: x >= 2].index

train_df = train_df[train_df['person_id'].isin(valid_persons(train_df))]
val_df = val_df[val_df['person_id'].isin(valid_persons(val_df))]
test_df = test_df[test_df['person_id'].isin(valid_persons(test_df))]

In [78]:
#train_df.groupby('person_id')['matched_code'].nunique().describe()
#val_df.groupby('person_id')['matched_code'].nunique().describe()
test_df.groupby('person_id')['matched_code'].nunique().describe()

count    28731.000000
mean         3.814869
std          1.759112
min          2.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         16.000000
Name: matched_code, dtype: float64

**Check date consistency**

In [79]:
# Convert quarter string (e.g., "Q1 2020") to a pandas timestamp (first day of quarter)
def quarter_to_timestamp(q):
    if pd.isna(q):
        return np.nan
    try:
        quarter, year = q.split()
        year = int(year)
        q_num = int(quarter[1])
        month = (q_num - 1) * 3 + 1
        return pd.Timestamp(year=year, month=month, day=1)
    except:
        return np.nan

In [80]:
train_df['start_ts'] = train_df['start_date'].apply(quarter_to_timestamp)
train_df['end_ts'] = train_df['end_date'].apply(quarter_to_timestamp)

val_df['start_ts'] = val_df['start_date'].apply(quarter_to_timestamp)
val_df['end_ts'] = val_df['end_date'].apply(quarter_to_timestamp)

test_df['start_ts'] = test_df['start_date'].apply(quarter_to_timestamp)
test_df['end_ts'] = test_df['end_date'].apply(quarter_to_timestamp)

In [81]:
train_df[['start_date', 'end_date', 'start_ts', 'end_ts']].sample(10)

,start_date,end_date,start_ts,end_ts
485542,Q1 2004,Q4 2006,2004-01-01,2006-10-01
399608,Q3 2007,Q1 2013,2007-07-01,2013-01-01
1199406,Q1 2009,Q4 2011,2009-01-01,2011-10-01
410153,Q1 2009,Q4 2014,2009-01-01,2014-10-01
423536,Q2 2011,Q3 2011,2011-04-01,2011-07-01
1334595,Q1 2003,Q4 2004,2003-01-01,2004-10-01
377359,Q1 1995,Q4 1996,1995-01-01,1996-10-01
487350,None,None,NaT,NaT
230577,None,None,NaT,NaT
721549,Q1 2012,Q3 2013,2012-01-01,2013-07-01


In [82]:
print("Invalid rate train: ", (train_df['start_ts'] > train_df['end_ts']).mean())
print("Invalid rate val: ", (val_df['start_ts'] > val_df['end_ts']).mean())
print("Invalid rate test: ", (test_df['start_ts'] > test_df['end_ts']).mean())

Invalid rate train:  1.6440353912465752e-05
Invalid rate val:  1.5473289234459014e-05
Invalid rate test:  1.5443896186129837e-05


Quarter-level dates are converted to timestamps for temporal ordering. Temporal inconsistencies affect fewer than 0.01% of records across all splits and can be tolerated. Career sequences should be ordered by start date, with end date used as a fallback.

**Add ESCO Skills**

In [ ]:
# Build mapping from occupation labels to lists of essential ESCO skills
occ_to_skills_cv = (
    essential_skills
    .groupby('occupationLabel')['skillLabel'] # group skills by occupation
    .apply(list) # aggregate skills into lists
    .reset_index(name='essential_skills') # convert to DataFrame
)

In [ ]:
# Normalize occupation labels to enable consistent merging (lowercase + strip)
occ_to_skills_cv["matched_label"] = occ_to_skills_cv["occupationLabel"].str.lower().str.strip()
occ_to_skills_cv.head()

,occupationLabel,essential_skills,matched_label
0,3D animator,"[particle animation, principles of animation, ...",3d animator
1,3D modeller,"[3D texturing, 3D lighting, augmented reality,...",3d modeller
2,3D printing technician,"[3D printing process, printing on large scale ...",3d printing technician
3,ATM repair technician,"[mechanical tools, security threats, ATM syste...",atm repair technician
4,EU funds manager,"[urban planning law, project management princi...",eu funds manager


In [ ]:
# Normalize occupation labels in all datasets to match ESCO format
train_df["matched_label"] = train_df["matched_label"].str.lower().str.strip()
val_df["matched_label"] = val_df["matched_label"].str.lower().str.strip()
test_df["matched_label"] = test_df["matched_label"].str.lower().str.strip()

In [86]:
# Attach essential skills to each observation via left join on normalized labels
train_df = train_df.merge(occ_to_skills_cv, on='matched_label', how='left')
val_df = val_df.merge(occ_to_skills_cv, on='matched_label', how='left')
test_df = test_df.merge(occ_to_skills_cv, on='matched_label', how='left')

In [87]:
train_df.head()

,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies,start_ts,end_ts,occupationLabel,essential_skills
0,0,resource manager,Resource managers manage resources for all pot...,1324.8.3,Q1 2016,Q2 2019,True,2016-01-01,2019-04-01,resource manager,"[corporate social responsibility, project mana..."
1,0,health and safety officer,Health and safety officers execute plans for t...,2263.3,Q1 2017,Q2 2019,True,2017-01-01,2019-04-01,health and safety officer,"[health, safety and hygiene legislation, techn..."
2,0,integration engineer,Integration engineers develop and implement so...,2511.17,Q1 2013,Q1 2016,True,2013-01-01,2016-01-01,integration engineer,"[procurement of ICT network equipment, inter-o..."
3,0,programme manager,Programme managers coordinate and oversee seve...,1213.4,Q2 2012,Q1 2013,True,2012-04-01,2013-01-01,programme manager,"[program management, corporate social responsi..."
4,0,product development engineering drafter,Product development engineering drafters desig...,3118.3.12,Q1 2011,Q2 2012,True,2011-01-01,2012-04-01,product development engineering drafter,"[engineering principles, circular economy, des..."


<a class="anchor" id="sub-section-5_2"></a>

## 5.2. Constructing CVs

</a>

In [88]:
# Remove consecutive duplicate occupations + preserve alignment between occupation codes, labels, and skill lists
def dedup_consecutive_aligned(codes, labels, skills):
    out_codes, out_labels, out_skills = [], [], []
    prev_code = object()

    for c, l, s in zip(codes, labels, skills):
        if c != prev_code:
            out_codes.append(c)
            out_labels.append(l)
            out_skills.append(s)
        prev_code = c

    return out_codes, out_labels, out_skills

In [89]:
# Build person-level career timelines from chronologically ordered job records
def build_timelines(df, remove_consecutive_duplicates=True):
    df = df.copy()

    # Ensure skill entries are always lists to keep sequence structure consistent
    df["essential_skills"] = df["essential_skills"].apply(lambda x: x if isinstance(x, list) else [])

    # Sort job records chronologically within each person
    df = df.sort_values(by=["person_id", "start_ts", "end_ts"], ascending=[True, True, True], na_position="last",)

    # Aggregate experiences into aligned occupation and skill sequences per person
    timelines = (
        df.groupby("person_id")
          .apply(lambda g: pd.Series({
              "occupation_codes": g["matched_code"].tolist(),
              "occupation_labels": g["matched_label"].tolist(),
              "essential_skills": g["essential_skills"].tolist(),  # list-of-lists (aligned to experiences)
          }))
          .reset_index()
    )

    # Optionally remove repeated consecutive occupations from each sequence
    if remove_consecutive_duplicates:
        timelines[["occupation_codes", "occupation_labels", "essential_skills"]] = timelines.apply(
            lambda r: pd.Series(dedup_consecutive_aligned(
                r["occupation_codes"], r["occupation_labels"], r["essential_skills"]
            )),
            axis=1
        )

    return timelines

In [90]:
train_timelines = build_timelines(train_df)
val_timelines = build_timelines(val_df)
test_timelines = build_timelines(test_df)

/var/folders/9w/mdnjt1210r561wjsryv639v40000gn/T/ipykernel_10276/1094813169.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/var/folders/9w/mdnjt1210r561wjsryv639v40000gn/T/ipykernel_10276/1094813169.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/var/folders/9w/mdnjt1210r561wjsryv639v40000gn/T/ipykernel_10276/1094813169.py:14: FutureWarning: DataFr

In [91]:
# Ensure all timeline sequences are aligned (same length across codes, labels, and skills)
assert (
    (train_timelines["occupation_codes"].apply(len)
     == train_timelines["occupation_labels"].apply(len))
    &
    (train_timelines["occupation_codes"].apply(len)
     == train_timelines["essential_skills"].apply(len))
).all()

# Timeline lengths
train_timelines["occupation_codes"].apply(len).describe()

count    229760.000000
mean          4.101227
std           2.027820
min           2.000000
25%           3.000000
50%           4.000000
75%           5.000000
max          22.000000
Name: occupation_codes, dtype: float64

In [92]:
train_timelines.sample(5)[['person_id', 'occupation_codes', 'occupation_labels', 'essential_skills']]

,person_id,occupation_codes,occupation_labels,essential_skills
188783,365786,"[7413.1, 9329.1, 7411.1, 7412.3, 3113.1]","[electricity distribution technician, factory ...","[[transmission towers, electrical discharge, e..."
181759,352038,"[1330.9, 1411.1, 1330.7, 1420.3, 1221.3.2.1, 3...","[telecommunications manager, accommodation man...","[[ICT communications protocols, telecom regula..."
108908,210910,"[3115.1.16, 3113.1.2, 8332.2, 7421.2]","[production engineering technician, electromec...","[[production processes, engineering principles..."
131229,254079,"[9329.1, 5223.4, 5223.6, 5141.1.1]","[factory hand, sales assistant, shop assistant...","[[cleaning products, industrial tools, cleanin..."
207595,402085,"[5223.6, 9321.2, 3343.4]","[shop assistant, hand packer, management assis...","[[product comprehension, company policies, app..."


<a class="anchor" id="sub-section-4_3"></a>

## 5.3. Constructing CV-Target Occupation pairs

</a>

In [93]:
# Flatten a list of skill lists and remove duplicates while preserving first occurrence order
def _flatten_dedup(list_of_lists):
    seen = set()
    out = []
    for lst in list_of_lists:
        if not isinstance(lst, list):
            continue
        for x in lst:
            if x not in seen:
                seen.add(x)
                out.append(x)
    return out

In [94]:
# Convert a sequence of past occupations and aligned skills into CV-style text
def history_to_cv_text(labels, skills_lol=None, max_roles=None, max_skills=None, skills_per_role=4):
    """
    labels: list of occupation labels (history)
    skills_lol: list-of-lists of skills aligned to labels (history)
    max_roles: keep only last k roles (None = keep all)
    max_skills: global cap after dedup (None = keep all)
    skills_per_role: keep only first k skills per occupation (None = keep all)
    """
    
    # Keep only the most recent roles if a history limit is specified
    if max_roles is not None:
        labels = labels[-max_roles:]
        if skills_lol is not None:
            skills_lol = skills_lol[-max_roles:]

    text = "Work experience: " + "; ".join(labels)

    if skills_lol is not None:
        # Limit the number of skills retained per occupation before flattening
        if skills_per_role is not None:
            skills_lol = [
                (lst[:skills_per_role] if isinstance(lst, list) else [])
                for lst in skills_lol
            ]
        # Flatten aligned skill lists and remove repeated skills across roles
        skills = _flatten_dedup(skills_lol)

        # Optionally cap the total number of skills included
        # Often unnecessary if max_roles*skills_per_role <= 40
        if max_skills is not None:
            skills = skills[:max_skills]

        # Append deduplicated skills if any remain
        if len(skills) > 0:
            text += ". Skills: " + "; ".join(skills)

    return text

In [95]:
# Build positive transition examples from person-level career timelines
def make_positive_pairs(timelines_df, max_history_roles=None, max_history_skills=None, skills_per_role=4):
    """
    timelines_df: DataFrame with person-level career sequences,
    containing occupation codes, labels, and aligned skill lists.
    """
    rows = []

    for _, r in timelines_df.iterrows():
        pid = r["person_id"]
        codes = r["occupation_codes"]
        labels = r["occupation_labels"]
        skills_lol = r["essential_skills"]

        L = len(codes)
        if L < 2:
            continue # should not occur after earlier filtering

        # Create one positive example for each observed next-step transition
        for t in range(1, L):
            cv_codes = codes[:t]
            cv_labels = labels[:t]
            cv_skills_lol = skills_lol[:t] # aligned skill history up to step t

            rows.append({
                "person_id": pid,
                "t": t,
                "cv_codes": cv_codes,
                "cv_labels": cv_labels,
                "cv_skills_lol": cv_skills_lol, # keeps step-aligned skills
                "cv_text": history_to_cv_text(
                    cv_labels,
                    skills_lol=cv_skills_lol,
                    max_roles=max_history_roles,
                    max_skills=max_history_skills,
                    skills_per_role=skills_per_role
                ),
                "target_code": codes[t],
                "target_label": labels[t],
            })

    return pd.DataFrame(rows)

In [96]:
train_pairs_pos = make_positive_pairs(train_timelines, max_history_roles=5, max_history_skills=20, skills_per_role=4)
val_pairs_pos = make_positive_pairs(val_timelines, max_history_roles=5, max_history_skills=20, skills_per_role=4)
test_pairs_pos = make_positive_pairs(test_timelines, max_history_roles=5, max_history_skills=20, skills_per_role=4)

In [97]:
len(train_pairs_pos), len(val_pairs_pos), len(test_pairs_pos)

(712538, 89135, 89214)

In [98]:
train_pairs_pos.sample(5)[["cv_text", "target_label"]]

,cv_text,target_label
255056,Work experience: supply chain manager; executi...,technical sales representative in electronic a...
450379,Work experience: bookkeeper; administrative as...,bookkeeper
349156,Work experience: bartender; sales assistant. S...,warehouse order picker
265755,Work experience: nanny; cashier; hospitality e...,tour organiser
652971,Work experience: factory hand; sales assistant...,road maintenance worker


In [99]:
train_pairs_pos.sample(5)

,person_id,t,cv_codes,cv_labels,cv_skills_lol,cv_text,target_code,target_label
699449,436289,2,"[5244.1, 3343.1]","[call centre agent, administrative assistant]","[[characteristics of products, credit card pay...",Work experience: call centre agent; administra...,1439.1,call centre manager
16221,10093,1,[9333.8],[warehouse worker],[[types of packaging used in industrial shipme...,Work experience: warehouse worker. Skills: typ...,3512.1,ict help desk agent
678965,423721,2,"[3343.1, 5223.4]","[administrative assistant, sales assistant]","[[company policies, maintain statutory books, ...",Work experience: administrative assistant; sal...,3432.6,visual merchandiser
152761,95075,1,[9329.1],[factory hand],"[[cleaning products, industrial tools, cleanin...",Work experience: factory hand. Skills: cleanin...,5230.1,cashier
630783,394219,2,"[4311.2, 4413.1]","[sales support assistant, proofreader]","[[bookkeeping regulations, sales activities, p...",Work experience: sales support assistant; proo...,2432.9,public relations officer


In [100]:
print("Empty CV in train: ", (train_pairs_pos["cv_text"].str.len() == 0).mean())
print("Empty CV in val: ", (val_pairs_pos["cv_text"].str.len() == 0).mean())
print("Empty CV in test: ", (test_pairs_pos["cv_text"].str.len() == 0).mean())

Empty CV in train:  0.0
Empty CV in val:  0.0
Empty CV in test:  0.0


<a class="anchor" id="sub-section-5_4"></a>

## 5.4. Merge CV-Target Occupation with JDs

</a>

In [101]:
train_pairs_pos[["target_code", "target_label"]].head()

,target_code,target_label
0,3512.1,ict help desk agent
1,3512.2,ict help desk manager
2,4222.1,customer contact centre information clerk
3,1324.4,move manager
4,3118.3.12,product development engineering drafter


In [102]:
jd_df[["occupationUri", "iscoGroup", "preferredLabel"]].head()

,occupationUri,iscoGroup,preferredLabel
0,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director
1,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator
2,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector
3,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician
4,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager


In [103]:
# Checking if there are no null values
jd_df["essential_skills"].isna().sum()

np.int64(0)

The JobHop dataset may be aligned with an earlier version of the ESCO taxonomy (v1.x), while this work uses a more recent release to construct job descriptions and derive ontology-based features. As a result, direct matching via ESCO occupation identifiers is not always possible due to changes in identifiers and labeling conventions.

To address this, JobHop occupations are mapped to ESCO using normalized string matching on occupation labels. This accounts for lexical differences across versions while preserving semantic alignment. Although this may introduce minor matching noise, such discrepancies are inherent to cross-version taxonomy integration and are applied consistently.

In [ ]:
# Normalize target occupation labels across all positive pairs
train_pairs_pos["target_label"] = train_pairs_pos["target_label"].str.lower().str.strip()
val_pairs_pos["target_label"] = val_pairs_pos["target_label"].str.lower().str.strip()
test_pairs_pos["target_label"] = test_pairs_pos["target_label"].str.lower().str.strip()

# Normalize occupation labels in job description
jd_df["preferredLabel"] = jd_df["preferredLabel"].str.lower().str.strip()

Due to label-based alignment between JobHop and ESCO, some occupation labels matched multiple ESCO entries. In such cases, a single representative job description was retained per label.

In [105]:
# Keep one job description per occupation label
jd_df_unique = jd_df.drop_duplicates(subset=["preferredLabel"], keep="first")

In [106]:
# Merge positive transition pairs with target occupation job descriptions
def merge_with_jd(pairs_pos, jd_df_unique):
    
    # Attach target occupation text and metadata from the ESCO job description table
    merged = pairs_pos.merge(
        jd_df_unique[["preferredLabel", "jd_text", "occupationUri", "iscoGroup"]],
        left_on="target_label",
        right_on="preferredLabel",
        how="inner"
    )

    # Ensure one row per observed transition after merging
    merged = merged.drop_duplicates(subset=["person_id", "t", "target_label"])

    return merged

In [107]:
train_pairs = merge_with_jd(train_pairs_pos, jd_df_unique)
val_pairs = merge_with_jd(val_pairs_pos, jd_df_unique)
test_pairs = merge_with_jd(test_pairs_pos, jd_df_unique)

In [108]:
print("Train: ", len(train_pairs) / len(train_pairs_pos))
print("Val: ", len(val_pairs) / len(val_pairs_pos))
print("Test: ", len(test_pairs) / len(test_pairs_pos))

Train:  0.9931161566119983
Val:  0.9927525663319684
Test:  0.9935996592463067


A small fraction of transitions (<1%) could not be matched to ESCO job descriptions and were excluded.

In [109]:
train_pairs.sample(3)[["cv_text", "target_label", "jd_text"]]

,cv_text,target_label,jd_text
269346,Work experience: building construction worker;...,welder,welder. Welders operate welding equipment in o...
213402,Work experience: building construction worker;...,metal products assembler,metal products assembler. Metal products assem...
697903,Work experience: ict help desk agent; pharmacy...,waste broker,waste broker. Waste brokers act as mediating p...


In [111]:
train_pairs.head(1)

,person_id,t,cv_codes,cv_labels,cv_skills_lol,cv_text,target_code,target_label,preferredLabel,jd_text,occupationUri,iscoGroup
0,0,1,[2353.1],[language school teacher],"[[assessment processes, learning difficulties,...",Work experience: language school teacher. Skil...,3512.1,ict help desk agent,ict help desk agent,ICT help desk agent. ICT help desk agents prov...,http://data.europa.eu/esco/occupation/aaeec9a7...,3512


In [112]:
pd.set_option("display.max_colwidth", None)
train_pairs["cv_text"].head()

0                                                                                                                                                                                                                                                                                                                                                                                                 Work experience: language school teacher. Skills: assessment processes; learning difficulties; instructional strategies; language teaching methods
1                                                                                                                                                                                                                                                             Work experience: language school teacher; ict help desk agent. Skills: assessment processes; learning difficulties; instructional strategies; language teaching methods; characteristics of products; ICT system user

Each dataset instance represents a career transition, pairing an accumulated CV history with a target ESCO-aligned job description, enriched with occupation hierarchy metadata for ontology-aware learning and explainability.

| Column                | Type               | Description |
|----------------------|--------------------|-------------|
| person_id            | Integer            | Anonymized individual identifier. Each person may appear in multiple rows, one per observed transition. |
| t                    | Integer            | Temporal index of the transition within an individual’s ordered career sequence. |
| cv_codes             | List[String]       | Chronologically ordered occupation codes representing work history up to time t (ISCO/ESCO-aligned). |
| cv_labels            | List[String]       | Human-readable occupation labels corresponding to cv_codes, preserving chronological order. |
| cv_skills_lol        | List[List[String]] | Essential ESCO skill labels associated with each past occupation up to time t (one list per occupation). |
| cv_text              | String             | Serialized textual representation of the candidate profile up to time t, combining occupation labels and essential skills. Used as input to the language model. |
| target_code          | String             | Occupation code of the next observed job (positive target). |
| target_label         | String             | Human-readable label of the target occupation. |
| preferredLabel_key   | String             | ESCO preferred occupation label used to align JobHop occupations with ESCO job descriptions. |
| jd_text              | String             | Textual job description constructed from ESCO occupation descriptions and essential skills. Used as job input to the language model. |
| occupationUri        | String             | Unique ESCO identifier of the occupation concept. |
| iscoGroup            | String             | ISCO group code (major or sub-major level), used for hierarchical analysis and ontology-aware negative sampling. |

In [113]:
def summarize_split(df, name):
    summary = {
        "split": name,
        "num_instances": len(df),
        "num_individuals": df["person_id"].nunique() if "person_id" in df.columns else None,
        "num_unique_target_occupations": df["occupationUri"].nunique() if "occupationUri" in df.columns else None,
        "num_unique_transitions": len(df),  # 1 row = 1 transition
    }
    return summary

In [114]:
train_summary = summarize_split(train_pairs, "train")
val_summary = summarize_split(val_pairs, "validation")
test_summary = summarize_split(test_pairs, "test")

split_stats = pd.DataFrame([train_summary, val_summary, test_summary])
display(split_stats)

,split,num_instances,num_individuals,num_unique_target_occupations,num_unique_transitions
0,train,707633,229333,2912,707633
1,validation,88489,28686,2501,88489
2,test,88643,28667,2504,88643


In [115]:
all_pairs = pd.concat([train_pairs, val_pairs, test_pairs], ignore_index=True)

overall_summary = {
    "total_instances": len(all_pairs),
    "total_individuals": all_pairs["person_id"].nunique(),
    "total_unique_target_occupations": all_pairs["occupationUri"].nunique(),
    "total_transitions": len(all_pairs),
}

print("\nOverall Dataset Summary:")
for k, v in overall_summary.items():
    print(f"{k}: {v}")


Overall Dataset Summary:
total_instances: 884765
total_individuals: 286686
total_unique_target_occupations: 2927
total_transitions: 884765


In [116]:
avg_transitions = all_pairs.groupby("person_id").size().mean()
print(f"\nAverage transitions per individual: {avg_transitions:.2f}")


Average transitions per individual: 3.09


In [118]:
train_ids = set(train_pairs["person_id"])
val_ids = set(val_pairs["person_id"])
test_ids = set(test_pairs["person_id"])

print("Train-Val overlap:", len(train_ids & val_ids))
print("Train-Test overlap:", len(train_ids & test_ids))
print("Val-Test overlap:", len(val_ids & test_ids))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


<a class="anchor" id="chapter6"></a>

# 6. Saving Dataset

</a>

In [99]:
os.getcwd()

'/Users/chloedeschanel/Documents/GitHub/Resume-Matching-Thesis/Notebooks'

In [ ]:
os.chdir('/Users/chloedeschanel/Documents/GitHub/Resume-Matching-Thesis/Data') # set directory

In [ ]:
#train_pairs.to_csv("train.csv", index=False)
#val_pairs.to_csv("val.csv", index=False)
#test_pairs.to_csv("test.csv", index=False)